# Complete RAG Course: From Zero to RAG Chatbot

**Duration:** 4 Hours  
**Level:** Beginner-Friendly  
**Goal:** Build a complete end-to-end RAG system with a PDF chatbot

---

## Course Overview

By the end of this notebook, you will:
- Understand why RAG (Retrieval-Augmented Generation) exists
- Learn how to extract text from PDFs
- Master three chunking strategies
- Convert text to embeddings using OpenAI
- Build a retrieval system using similarity search
- Generate grounded answers using LLMs
- Create a complete PDF chatbot

Let's begin!

---

# SECTION 1: WHY RAG EXISTS

## The Problem

Most beginners think LLMs know everything. The reality is different:

### Reality: LLMs Don't Know Your Company Documents

Consider this question:

**Question:** "What is the notice period for interns?"

This answer exists inside your company's PDF handbook, but the LLM has no access to it. The model was trained on publicly available data and doesn't know about your private documents.

**The Problem RAG Solves:**
1. LLM has no context about your PDF
2. User asks a question
3. LLM guesses or hallucinates
4. User gets incorrect or irrelevant answers

RAG bridges this gap by providing context to the LLM before asking the question.

### Step 1: Create OpenAI Client

**Why we're doing it:**
- We need to communicate with OpenAI's API to use embeddings and chat models

**What we expect:**
- A working OpenAI client that can make API calls

**Key takeaway:**
- Always initialize the OpenAI client first before making any API calls

In [36]:
from openai import OpenAI
import os
import warnings

# Suppress all warnings globally for cleaner output
warnings.filterwarnings('ignore')

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

### Step 2: Ask WITHOUT RAG (The Problem)

**Why we're doing it:**
- To demonstrate what happens when we ask without providing context

**What we expect:**
- The model will respond, but it's just guessing since it has no knowledge of your company handbook

**Key takeaway:**
- Without context (RAG), LLMs cannot answer questions about private documents

In [37]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "What is the notice period for interns?"
        }
    ]
)

print(response.choices[0].message.content)

The notice period for interns can vary depending on the company or organization, the specific internship program, and the terms outlined in the internship agreement. Generally, it can range from a few days to two weeks. Some companies might not have a formal notice period, especially for short internships. It's always best for interns to check their internship agreement or consult with their supervisor or HR department for specific policies regarding notice periods.


**Observation:**

The model responded, but it has no idea about your company's actual policy. It's making an educated guess based on general knowledge. This is the fundamental problem RAG solves.

**What's missing:**
- Access to your company handbook
- Grounded, factual information
- Accuracy guarantee

### Step 3: Ask WITH Manual Context (The Solution)

**Why we're doing it:**
- To show that providing context gives us accurate answers

**What we expect:**
- When we explicitly provide the answer in context, the model answers accurately

**Key takeaway:**
- Context + Question = Accurate Grounded Answer

In [39]:
context = """
Interns must serve a notice period of 15 days before leaving the organization.
"""

question = "What is the notice period for interns?"

prompt = f"""
Answer using the provided context.

Context:
{context}

Question:
{question}
"""

print(prompt)


Answer using the provided context.

Context:

Interns must serve a notice period of 15 days before leaving the organization.


Question:
What is the notice period for interns?



In [3]:
context = """
Interns must serve a notice period of 15 days before leaving the organization.
"""

question = "What is the notice period for interns?"

prompt = f"""
Answer using the provided context.

Context:
{context}

Question:
{question}
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

The notice period for interns is 15 days.


**Observation:**

Now the model gives an accurate answer: **15 days**.

This is the essence of RAG.

---

## The RAG Pipeline

```
Question
    ↓
[Search in Documents]
    ↓
Retrieved Context
    ↓
Question + Context
    ↓
[Pass to LLM]
    ↓
Grounded Answer
```

In the next sections, we'll build each component of this pipeline.

---

# SECTION 2: PDF INGESTION

## Converting PDF to Text

Most documents exist as PDFs. Before we can process them, we need to extract the text.

**Why extraction is necessary:**
- PDFs are formatted binary files, not plain text
- We need raw text to chunk, embed, and search
- Direct embedding of PDF binary data won't work

**Tool we'll use:**
- `pypdf` library to read and extract text from PDFs

### Extract Text from employee_policy_handbook.pdf

**Why we're doing it:**
- To convert the PDF file into usable text format

**What we expect:**
- Raw text extracted from all pages of the PDF

**Key takeaway:**
- `PdfReader` from pypdf makes PDF extraction simple

In [54]:
pip install pypdf


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [40]:
from pypdf import PdfReader

reader = PdfReader(
    "employee_policy_handbook.pdf"
)

full_text = ""

for page in reader.pages:
    full_text += page.extract_text()

print(f"Total characters extracted: {len(full_text)}")
print("\nFirst 2000 characters:")
print(full_text[:2000])

Total characters extracted: 3165

First 2000 characters:
Employee  Policy  Handbook  
Section  1:  Employment  Categories  
The  company  has  three  employment  categories:  
1.  Full-Time  Employee  2.  Intern  3.  Contract  Employee  
Each  category  follows  different  employment  rules  and  exit  procedures.  
 
Section  2:  Notice  Period  Policy  
Full-time  employees  must  serve  a  notice  period  of  60  days  before  resignation.  
Interns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.  
Contract  employees  follow  project-specific  exit  timelines  determined  by  their  respective  project  
managers.
 
Failure  to  complete  the  required  notice  period  may  result  in  delays  in  final  settlement  
processing.
 
 
Section  3:  Laptop  Return  Policy  
All  employees  must  return  company-issued  laptops  before  their  final  working  day.  
Interns  must  submit  laptops  to  the  IT  department  at  least  2  days  before  th

### Observation

We now have the entire handbook as raw text. But there's a critical question:

**Can we embed the entire document directly?**

No. Here's why:
1. **Context Window Limits:** LLM context windows are limited (even GPT-4 has finite limits)
2. **Embedding Efficiency:** Larger texts create less precise embeddings
3. **Retrieval Precision:** We need to find specific relevant sections, not search entire documents

**Solution:** Split the document into smaller chunks.

This introduces us to the concept of **chunking** – which we'll cover in detail in Section 5.

---

# SECTION 3: WHAT ARE EMBEDDINGS?

## Understanding Embeddings: From Text to Numbers

Embeddings are the core technology that makes RAG work. Let's understand them before diving into chunking.

### The Keyword Search Problem

**Example:**

Document: *"Interns must serve a notice period of 15 days before leaving."*

Question: *"How long before resigning should interns inform the company?"*

**Human understanding:**
- We immediately see the connection
- "notice period" = "how long to inform"
- Answer is clearly in the document

**Computer understanding (without embeddings):**
- Keyword search looks for exact words
- "notice period" ≠ "how long to inform"
- Computer says: "No match found"

This is the core problem embeddings solve: **semantic understanding through mathematics**.

---

## What Are Embeddings?

**Simple definition:**

Embeddings convert text into a list of numbers (called a vector) where:
- Similar meanings → Similar numbers
- Different meanings → Different numbers

**Visual representation:**
```
Text: "Interns must serve a notice period of 15 days."
   ↓
Embedding: [0.024, -0.156, 0.834, 0.421, ...1536 numbers total...]
```

**Why 1536 dimensions?**
- OpenAI's `text-embedding-3-small` model produces 1536-dimensional vectors
- More dimensions = more complex semantic information captured
- We don't manually interpret these numbers; the model does the interpretation for us

---

## Key Insight

**We do NOT manually interpret embeddings.**

We use them to:
1. Compare meaning mathematically (similarity)
2. Find semantically related text
3. Retrieve relevant context

The actual numbers are abstract representations of meaning – and that's exactly what we need for RAG.

### Step 1: Create an Embedding

**Why we're doing it:**
- To see what an embedding looks like
- To understand the structure of embedding responses from OpenAI

**What we expect:**
- A vector of 1536 numbers representing the semantic meaning of our text

**Key takeaway:**
- OpenAI's API returns embeddings as lists of floats

In [41]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Interns must serve a notice period of 15 days."
)

embedding = response.data[0].embedding

print(f"Type: {type(embedding)}")
print(f"Length (dimensions): {len(embedding)}")

Type: <class 'list'>
Length (dimensions): 1536


In [ ]:
type(embedding)

1536

### Step 2: Inspect the Embedding

**Why we're doing it:**
- To see the actual numbers in an embedding
- To understand that embeddings are just arrays of floats

**What we expect:**
- First 10 values of the embedding (they won't make intuitive sense – that's normal!)

**Key takeaway:**
- The numbers represent abstract semantic meaning; we don't interpret them manually

In [6]:
print("First 10 values of the embedding:")
print(embedding[:10])

print("\nLast 10 values:")
print(embedding[-10:])

First 10 values of the embedding:
[-0.048370361328125, 0.07501220703125, 0.029144287109375, -0.01519775390625, -0.035064697265625, 0.042083740234375, 0.026702880859375, 0.0237579345703125, 0.076171875, 0.017791748046875]

Last 10 values:
[0.0011234283447265625, 0.05889892578125, -0.01445770263671875, -0.0002856254577636719, 0.0022754669189453125, -0.01435089111328125, 0.0228424072265625, -0.014739990234375, -0.01430511474609375, 0.001750946044921875]


**Observation:**

These numbers don't have intuitive meaning to humans. But mathematically, they represent the semantic content of the text perfectly.

Now you know what embeddings are. In the next section, we'll learn how to **compare embeddings** to find similar text.

---

# SECTION 4: SIMILARITY SEARCH

## How to Compare Embeddings

Now that we can create embeddings, we need a way to compare them.

**The question:** How do we know if two embeddings represent similar meanings?

**The answer:** Cosine Similarity – a mathematical measure of similarity between vectors.

### What is Cosine Similarity?

**Simple explanation:**
- Cosine similarity measures the angle between two vectors
- Same direction (close angle) = high similarity (closer to 1)
- Opposite direction (wide angle) = low similarity (closer to 0)

**Range:**
- 1.0 = Identical meaning
- 0.5 = Somewhat similar
- 0.0 = Completely different

We'll use scikit-learn's `cosine_similarity` function to calculate this mathematically.

### Step 1: Create Multiple Embeddings

**Why we're doing it:**
- To have multiple texts to compare
- To demonstrate similarity between related and unrelated texts

**What we expect:**
- Three embeddings for three different texts

**Key takeaway:**
- We can batch embed multiple texts in a single API call for efficiency

In [47]:
texts = [
    "What is the intern notice period?",
    "How many days before leaving must interns inform?",
    "Cafeteria timings"
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

embeddings = [
    item.embedding
    for item in response.data
]

print(f"Created {len(embeddings)} embeddings")

Created 3 embeddings


### Step 2: Calculate Cosine Similarity

**Why we're doing it:**
- To compare the semantic similarity of different texts
- To demonstrate that similar texts have higher scores

**What we expect:**
- High similarity between questions 1 and 2 (both about intern notice period)
- Low similarity between question 1 and 3 (unrelated topics)

**Key takeaway:**
- Cosine similarity is the mathematical foundation of RAG retrieval

In [48]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import warnings

# Suppress ALL warnings from scikit-learn matrix operations
warnings.filterwarnings('ignore')

# Compare text 1 vs text 2 (both about notice period)
similarity_1 = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)[0][0]

# Compare text 1 vs text 3 (unrelated - cafeteria)
similarity_2 = cosine_similarity(
    [embeddings[0]],
    [embeddings[2]]
)[0][0]

print(f"'Notice period' vs 'How long to inform': {similarity_1:.4f}")
print(f"'Notice period' vs 'Cafeteria timings': {similarity_2:.4f}")

'Notice period' vs 'How long to inform': 0.6028
'Notice period' vs 'Cafeteria timings': 0.2292


**Observation:**

The first similarity score is significantly higher than the second. This is the **heart of RAG retrieval**.

**Why this matters:**
- When a user asks a question, we embed it
- We compare it against all chunks in our document
- The chunks with highest similarity are the most relevant
- We retrieve those and pass them to the LLM

This is exactly how RAG finds the right context for any question.

---

# SECTION 5: WHY CHUNKING EXISTS

## The Chunking Problem

Remember our extracted PDF text? It's one long string of characters.

**Can we just embed the entire document as one chunk?**

No. Here's why:

### Problem 1: Context Window Limits
- LLMs have finite context windows
- GPT-4o has ~128K tokens (large but not infinite)
- A 100-page handbook could easily exceed this

### Problem 2: Embedding Quality
- Embeddings of very long texts lose precision
- Longer texts have diluted semantic meaning
- Similarity search becomes less accurate

### Problem 3: Retrieval Precision
- If document A = 50 pages, it's one big blob
- A question about page 3 gets the entire 50-page context
- LLM can get confused with irrelevant information

**Solution: Split documents into smaller, meaningful chunks.**

---

## Three Chunking Strategies

In this course, we'll learn three approaches:

### Strategy 1: Fixed Size Chunking

**How it works:**
- Split text into chunks of exactly N characters
- Simple, predictable, consistent

**Pros:**
- Simple to implement
- Predictable chunk sizes
- Works quickly

**Cons:**
- May split sentences or paragraphs in the middle
- Chunk boundaries are arbitrary
- Loses semantic structure

**Example:**
```
Document: "Hello world. This is a test."
Chunk size: 10 characters

Chunk 1: "Hello worl"
Chunk 2: "d. This is"
Chunk 3: " a test."
```

See the problem? Chunks cut through sentences.

#### Implement Fixed Size Chunking

**Why we're doing it:**
- To understand the simplest chunking approach
- To see its strengths and weaknesses

**What we expect:**
- Document split into fixed-size chunks

**Key takeaway:**
- Fixed size chunking is fast but semantically poor

In [ ]:
text = full_text  # Use our extracted PDF text

chunk_size = 300
fixed_chunks = []

for i in range(0, len(text), chunk_size):
    fixed_chunks.append(text[i:i+chunk_size])

print(f"Total chunks: {len(fixed_chunks)}")
print(f"\nFirst chunk ({len(fixed_chunks[0])} chars):")
print(fixed_chunks[0])
print(f"\nSecond chunk ({len(fixed_chunks[1])} chars):")
print(fixed_chunks[1])

Total chunks: 11

First chunk (300 chars):
Employee  Policy  Handbook  
Section  1:  Employment  Categories  
The  company  has  three  employment  categories:  
1.  Full-Time  Employee  2.  Intern  3.  Contract  Employee  
Each  category  follows  different  employment  rules  and  exit  procedures.  
 
Section  2:  Notice  Period  Policy  


Second chunk (300 chars):

Full-time  employees  must  serve  a  notice  period  of  60  days  before  resignation.  
Interns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.  
Contract  employees  follow  project-specific  exit  timelines  determined  by  their  respective  project  
manage


### Strategy 2: Sentence Chunking

**How it works:**
- Split text by sentence boundaries (., !, ?)
- Each chunk is a complete sentence

**Pros:**
- Preserves complete sentences
- More semantic meaning per chunk
- Better than fixed-size for most documents

**Cons:**
- Chunk sizes vary significantly
- Some sentences may be too long
- Some sentences may be too short (fragments)

**Example:**
```
Document: "Hello world. This is a test. It works."

Chunk 1: "Hello world."
Chunk 2: "This is a test."
Chunk 3: "It works."
```

Much better! Complete sentences stay together.

#### Implement Sentence Chunking

**Why we're doing it:**
- To preserve semantic meaning at sentence boundaries
- To handle variable-length chunks naturally

**What we expect:**
- Document split by sentence boundaries

**Key takeaway:**
- Sentence chunking respects semantic units better than fixed-size

In [10]:
import re

text = full_text

# Split by sentence boundaries (., !, ?)
sentence_chunks = re.split(
    r'(?<=[.!?])\s+',
    text
)

print(f"Total chunks: {len(sentence_chunks)}")
print(f"\nFirst chunk:")
print(sentence_chunks[0])
print(f"\nSecond chunk:")
print(sentence_chunks[1])
print(f"\nThird chunk:")
print(sentence_chunks[2])

Total chunks: 42

First chunk:
Employee  Policy  Handbook  
Section  1:  Employment  Categories  
The  company  has  three  employment  categories:  
1.

Second chunk:
Full-Time  Employee  2.

Third chunk:
Intern  3.


In [52]:
sentence_chunks[5]

'Interns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.'

### Strategy 3: Recursive Chunking (Recommended)

**How it works:**
- Uses recursive splitting with multiple separators
- Default separators: paragraphs → lines → words → characters
- Adds chunk overlap for context continuity

**Why recursive?**
- Starts with large separators (paragraph breaks)
- If chunks are still too large, splits by lines
- If still too large, splits by words
- If still too large, splits by characters

This gives the best of both worlds: semantic structure + controlled size.

**Key parameters:**
- `chunk_size`: Maximum characters per chunk
- `chunk_overlap`: Characters repeated in adjacent chunks (for context continuity)

**Example with chunk_overlap:**
```
Chunk 1: "Interns notice period 15 days before leaving..."
Chunk 2: "...before leaving organization. Must return laptop..."
              ^^^^^^^^^^^^^^^
            (overlapping context)
```

**Pros:**
- Respects document structure
- Controlled chunk sizes
- Overlap prevents information loss at boundaries
- Standard in production RAG systems

**Cons:**
- Slightly more complex
- Requires LangChain library
- Overlap increases storage and compute

#### Implement Recursive Chunking

**Why we're doing it:**
- Recursive chunking is the industry standard
- It balances semantic meaning with chunk size control

**What we expect:**
- Document split using recursive separators
- Chunks that respect document structure

**Key takeaway:**
- Recursive chunking is the recommended strategy for production RAG systems

In [53]:
pip install langchain-text-splitters


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50, 
    # separators=[]
)

recursive_chunks = splitter.split_text(
    full_text
)

print(f"Total Chunks: {len(recursive_chunks)}")
print(f"\nFirst Chunk:")
print(recursive_chunks[0])
print(f"\nSecond Chunk:")
print(recursive_chunks[1])

Total Chunks: 13

First Chunk:
Employee  Policy  Handbook  
Section  1:  Employment  Categories  
The  company  has  three  employment  categories:  
1.  Full-Time  Employee  2.  Intern  3.  Contract  Employee  
Each  category  follows  different  employment  rules  and  exit  procedures.  
 
Section  2:  Notice  Period  Policy


Second Chunk:
Section  2:  Notice  Period  Policy  
Full-time  employees  must  serve  a  notice  period  of  60  days  before  resignation.  
Interns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.


In [61]:
print(recursive_chunks[0])

Employee  Policy  Handbook  
Section  1:  Employment  Categories  
The  company  has  three  employment  categories:  
1.  Full-Time  Employee  2.  Intern  3.  Contract  Employee  
Each  category  follows  different  employment  rules  and  exit  procedures.  
 
Section  2:  Notice  Period  Policy


### Observation

Notice how the recursive chunks:
1. Preserve complete sentences and paragraphs
2. Respect document structure
3. Have controlled size (around 300 chars)
4. Show overlap between chunks (some text repeats)

**Important Note:**
LangChain automatically tries different separators internally. For this course, we don't need to specify custom separators—the defaults work excellently for most documents.

In production, separators can be customized for specific document types (e.g., markdown, code, legal documents).

---

# SECTION 6: EMBEDDING ALL CHUNKS

## From Chunks to Searchable Vectors

Now that we have chunks, the next step is straightforward:
- Embed each chunk
- Store the embeddings
- Use them for similarity search

This is where embedding becomes powerful: we convert every piece of knowledge in our document into searchable vectors.

### Embed All Chunks

**Why we're doing it:**
- To convert our knowledge base (chunks) into searchable vectors
- To prepare for similarity search

**What we expect:**
- One embedding per chunk
- Same number of embeddings as chunks

**Key takeaway:**
- Batch embedding is efficient; we embed all chunks in one API call

In [12]:
chunks = recursive_chunks  # Use our recursive chunks

print(f"Embedding {len(chunks)} chunks...")

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunks
)

chunk_embeddings = [
    item.embedding
    for item in response.data
]

print(f"\n✓ Chunks: {len(chunks)}")
print(f"✓ Embeddings: {len(chunk_embeddings)}")
print(f"✓ Embedding dimensions: {len(chunk_embeddings[0])}")

Embedding 13 chunks...

✓ Chunks: 13
✓ Embeddings: 13
✓ Embedding dimensions: 1536


In [63]:
len(chunk_embeddings)

13

In [80]:
len(chunk_embeddings[0])

1536

**What we've accomplished:**

```
PDF Document
    ↓
Extracted Text
    ↓
Recursive Chunks (e.g., 50 chunks)
    ↓
Embeddings (50 vectors of 1536 dimensions each)
```

We now have a searchable knowledge base. When a user asks a question, we can:
1. Embed their question
2. Compare it against all chunk embeddings
3. Find the most similar chunks
4. Pass those to the LLM

This is the retrieval system of RAG.

---

# SECTION 7: RETRIEVAL

## Finding Relevant Context

Now comes the critical part: when a user asks a question, we need to find the most relevant chunks.

**The retrieval process:**
1. User asks a question
2. Embed the question
3. Compare against all chunk embeddings using cosine similarity
4. Retrieve the top-K most similar chunks
5. Pass these chunks as context to the LLM

This is the **R** in RAG – Retrieval.

### Step 1: Embed the User's Question

**Why we're doing it:**
- To represent the user's question in the same vector space as our chunks
- To enable similarity comparison

**What we expect:**
- A single embedding for the user's question

**Key takeaway:**
- Questions and documents are embedded using the same model, ensuring compatibility

In [13]:
query = "What is the notice period for interns?"

query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_embedding = (
    query_response.data[0].embedding
)

print(f"Query: {query}")
print(f"Embedding dimensions: {len(query_embedding)}")

Query: What is the notice period for interns?
Embedding dimensions: 1536


In [ ]:
text vs vector
vector vs vector

### Step 2: Calculate Similarity and Retrieve Top Chunks

**Why we're doing it:**
- To find the chunks most relevant to the user's question
- To rank chunks by relevance

**What we expect:**
- Top 3 chunks ranked by similarity score
- Highest similarity scores first

**Key takeaway:**
- Cosine similarity quantifies relevance; higher scores = better matches

In [76]:
scores=[]
for chunk, embedding in zip(
    chunks,
    chunk_embeddings
):
    score = cosine_similarity(
        [query_embedding],
        [embedding]
    )[0][0]

    scores.append(
        (chunk, score)
    )

In [77]:
scores

[('Employee  Policy  Handbook  \nSection  1:  Employment  Categories  \nThe  company  has  three  employment  categories:  \n1.  Full-Time  Employee  2.  Intern  3.  Contract  Employee  \nEach  category  follows  different  employment  rules  and  exit  procedures.  \n \nSection  2:  Notice  Period  Policy',
  0.5189245554362976),
 ('Section  2:  Notice  Period  Policy  \nFull-time  employees  must  serve  a  notice  period  of  60  days  before  resignation.  \nInterns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.',
  0.7052526043006707),
 ('Contract  employees  follow  project-specific  exit  timelines  determined  by  their  respective  project  \nmanagers.\n \nFailure  to  complete  the  required  notice  period  may  result  in  delays  in  final  settlement  \nprocessing.\n \n \nSection  3:  Laptop  Return  Policy',
  0.43892992104525974),
 ('Section  3:  Laptop  Return  Policy  \nAll  employees  must  return  company-issued  laptops  before  

In [30]:
scores = []

for chunk, embedding in zip(
    chunks,
    chunk_embeddings
):
    score = cosine_similarity(
        [query_embedding],
        [embedding]
    )[0][0]

    scores.append(
        (chunk, score)
    )

# Sort by score (highest first)
scores.sort(
    key=lambda x: x[1],
    reverse=True
)

# Get top 3 most relevant chunks
top_chunks = scores[:3]

print(f"Query: {query}")
print(f"\nTop 3 Most Relevant Chunks:\n")

for i, (chunk, score) in enumerate(top_chunks, 1):
    print(f"Chunk {i} (Similarity: {score:.4f})")
    print(chunk)


Query: What is the notice period for interns?

Top 3 Most Relevant Chunks:

Chunk 1 (Similarity: 0.7910)
Section  8:  Frequently  Asked  Questions  
Q:  What  is  the  notice  period  for  interns?  
A:  Interns  must  serve  a  15-day  notice  period.  
Q:  What  is  the  notice  period  for  full-time  employees?  
A:  Full-time  employees  must  serve  a  60-day  notice  period.


Chunk 2 (Similarity: 0.7053)
Section  2:  Notice  Period  Policy  
Full-time  employees  must  serve  a  notice  period  of  60  days  before  resignation.  
Interns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.


Chunk 3 (Similarity: 0.5910)
Interns  receive  6  leave  days  during  their  internship  period.  
Unused  leave  cannot  be  carried  forward  for  interns.  
Emergency  leave  requests  must  be  approved  by  the  reporting  manager.  
 
Section  7:  Exit  Process




In [72]:
top_chunks

[('Section  8:  Frequently  Asked  Questions  \nQ:  What  is  the  notice  period  for  interns?  \nA:  Interns  must  serve  a  15-day  notice  period.  \nQ:  What  is  the  notice  period  for  full-time  employees?  \nA:  Full-time  employees  must  serve  a  60-day  notice  period.',
  0.790978878191102),
 ('Section  2:  Notice  Period  Policy  \nFull-time  employees  must  serve  a  notice  period  of  60  days  before  resignation.  \nInterns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.',
  0.7052526043006707),
 ('Interns  receive  6  leave  days  during  their  internship  period.  \nUnused  leave  cannot  be  carried  forward  for  interns.  \nEmergency  leave  requests  must  be  approved  by  the  reporting  manager.  \n \nSection  7:  Exit  Process',
  0.590952553570901)]

In [66]:
scores[0]

('Section  8:  Frequently  Asked  Questions  \nQ:  What  is  the  notice  period  for  interns?  \nA:  Interns  must  serve  a  15-day  notice  period.  \nQ:  What  is  the  notice  period  for  full-time  employees?  \nA:  Full-time  employees  must  serve  a  60-day  notice  period.',
 0.790978878191102)

**What just happened:**

We successfully retrieved the most relevant chunks!

**Retrieval quality metrics:**
- Similarity scores tell us confidence
- High scores (>0.7) = very relevant
- Medium scores (0.5-0.7) = somewhat relevant
- Low scores (<0.5) = probably not relevant

**The key insight:**
Even though our question uses different words than the document, embeddings found the right chunks because they understand semantic meaning.

This is the power of RAG retrieval.

---

# SECTION 8: GENERATION

## From Retrieved Context to Final Answer

We have retrieved the most relevant chunks. Now comes the final step: use the LLM to generate a grounded answer.

This is the **G** in RAG – Generation.

**Process:**
1. Take the retrieved chunks
2. Combine them into context
3. Build a prompt with context + question
4. Send to LLM
5. Return the generated answer

### Step 1: Combine Retrieved Chunks into Context

**Why we're doing it:**
- To format retrieved chunks into a single context block
- To prepare context for the LLM prompt

**What we expect:**
- A formatted context string containing all top chunks

**Key takeaway:**
- Formatting context properly improves LLM understanding

In [73]:
context = "\n\n".join(
    chunk
    for chunk, score in top_chunks
)

print("Combined Context:")
print(context)

Combined Context:
Section  8:  Frequently  Asked  Questions  
Q:  What  is  the  notice  period  for  interns?  
A:  Interns  must  serve  a  15-day  notice  period.  
Q:  What  is  the  notice  period  for  full-time  employees?  
A:  Full-time  employees  must  serve  a  60-day  notice  period.

Section  2:  Notice  Period  Policy  
Full-time  employees  must  serve  a  notice  period  of  60  days  before  resignation.  
Interns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.

Interns  receive  6  leave  days  during  their  internship  period.  
Unused  leave  cannot  be  carried  forward  for  interns.  
Emergency  leave  requests  must  be  approved  by  the  reporting  manager.  
 
Section  7:  Exit  Process


In [75]:
type(context)

str

### Step 2: Build the Prompt

**Why we're doing it:**
- To create a prompt that tells the LLM to use only the provided context
- To ensure grounded, accurate answers

**What we expect:**
- A well-formatted prompt with clear context and question

**Key takeaway:**
- Good prompt structure = better LLM outputs

In [ ]:
query = "What is the notice period for interns?"

In [74]:
query = "What is the notice period for interns?"

prompt = f"""
Answer the question using ONLY the context provided.

Context:
{context}

Question:
{query}
"""

print("Prompt being sent to LLM:")
print(prompt)


Prompt being sent to LLM:

Answer the question using ONLY the context provided.

Context:
Section  8:  Frequently  Asked  Questions  
Q:  What  is  the  notice  period  for  interns?  
A:  Interns  must  serve  a  15-day  notice  period.  
Q:  What  is  the  notice  period  for  full-time  employees?  
A:  Full-time  employees  must  serve  a  60-day  notice  period.

Section  2:  Notice  Period  Policy  
Full-time  employees  must  serve  a  notice  period  of  60  days  before  resignation.  
Interns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.

Interns  receive  6  leave  days  during  their  internship  period.  
Unused  leave  cannot  be  carried  forward  for  interns.  
Emergency  leave  requests  must  be  approved  by  the  reporting  manager.  
 
Section  7:  Exit  Process

Question:
What is the notice period for interns?



### Step 3: Call the LLM

**Why we're doing it:**
- To generate a grounded answer based on retrieved context
- To complete the RAG pipeline

**What we expect:**
- A clear, accurate answer based on the provided context

**Key takeaway:**
- LLM generates answers grounded in the retrieved chunks, not from general knowledge

In [17]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response.choices[0].message.content

print("FINAL RAG ANSWER")
print(answer)


FINAL RAG ANSWER
Interns must serve a 15-day notice period.


**Congratulations!** 🎉

You've just built your first complete RAG system!

**The full pipeline:**
```
Question
    ↓
[Embed Question]
    ↓
[Search Similar Chunks]
    ↓
[Retrieve Top 3 Chunks]
    ↓
[Build Prompt with Context]
    ↓
[Call GPT-4o-mini]
    ↓
Grounded Answer
```

This answer is grounded in the actual document – not a guess.

---

# SECTION 9: PROMPT IMPROVEMENT

## Preventing Hallucinations

RAG significantly reduces hallucinations, but we can make it even better with prompt engineering.

**Current issue:**
- If context doesn't answer the question, the LLM might still try to guess

**Solution:**
- Explicitly instruct the LLM to say "Not found in context" when it can't answer

This is prompt engineering – tuning the instructions to get better results.

### Improved Prompt with Hallucination Prevention

**Why we're doing it:**
- To improve answer reliability
- To prevent the LLM from guessing when information isn't available

**What we expect:**
- Answers that are either grounded or explicitly state "Not found in context"

**Key takeaway:**
- Good prompt engineering makes RAG systems more reliable

In [18]:
prompt = f"""
You are an HR policy assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context,
respond with: Not found in context.

Context:
{context}

Question:
{query}
"""

print("Improved Prompt:")
print(prompt)


Improved Prompt:

You are an HR policy assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context,
respond with: Not found in context.

Context:
Section  8:  Frequently  Asked  Questions  
Q:  What  is  the  notice  period  for  interns?  
A:  Interns  must  serve  a  15-day  notice  period.  
Q:  What  is  the  notice  period  for  full-time  employees?  
A:  Full-time  employees  must  serve  a  60-day  notice  period.

Section  2:  Notice  Period  Policy  
Full-time  employees  must  serve  a  notice  period  of  60  days  before  resignation.  
Interns  must  serve  a  notice  period  of  15  days  before  leaving  the  organization.

Interns  receive  6  leave  days  during  their  internship  period.  
Unused  leave  cannot  be  carried  forward  for  interns.  
Emergency  leave  requests  must  be  approved  by  the  reporting  manager.  
 
Section  7:  Exit  Process

Question:
What is the notice period for interns?



### Call LLM with Improved Prompt

**Why we're doing it:**
- To demonstrate how better prompts improve RAG reliability

**What we expect:**
- Same answer quality but with better guardrails

**Key takeaway:**
- Prompt wording significantly affects LLM behavior

In [19]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

Interns must serve a 15-day notice period.


---

# SECTION 10: TEST QUESTIONS

## Testing the RAG System

Let's test our RAG system with various questions to see how well it performs.

We'll test:
1. Questions answerable from the handbook
2. Questions outside the handbook scope
3. Complex multi-part questions

### Test Question 1: "Do interns receive health insurance?"

**Expected Answer:** No (or "Not found in context" if not covered)

**Why this test:**
- Tests if RAG correctly retrieves negative answers
- Or admits when information isn't available

In [32]:
def ask_rag_question(question):
    """Helper function to ask RAG questions"""
    # Embed question
    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    )
    query_embedding = query_response.data[0].embedding
    
    # Find similar chunks
    scores = []
    for chunk, embedding in zip(chunks, chunk_embeddings):
        score = cosine_similarity(
            [query_embedding],
            [embedding]
        )[0][0]
        scores.append((chunk, score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    top_chunks = scores[:3]
    
    # Build context
    context = "\n\n".join(chunk for chunk, score in top_chunks)
    
    # Build prompt
    prompt = f"""
You are an HR policy assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context,
respond with: Not found in context.

Context:
{context}

Question:
{question}
"""
    
    # Get answer
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

question = "Do interns receive health insurance?"
answer = ask_rag_question(question)

print(f"Question: {question}")
print(f"Answer: {answer}")

Question: Do interns receive health insurance?
Answer: Not found in context.


### Test Question 2: "When should interns return laptops?"

**Expected Answer:** 2 days before exit

**Why this test:**
- Tests specific information retrieval
- Should work well if laptop return policy is in the document

In [33]:
question = "When should interns return laptops?"
answer = ask_rag_question(question)

print(f"Question: {question}")
print(f"Answer: {answer}")

Question: When should interns return laptops?
Answer: Interns must return laptops at least 2 days before their exit date.


### Test Question 3: "What must interns do before leaving?"

**Expected Answer:**
- Notice period
- Laptop return
- Exit interview
- Manager approval

**Why this test:**
- Tests complex multi-part questions
- Tests if RAG can synthesize multiple chunks

In [34]:
question = "What must interns do before leaving?"
answer = ask_rag_question(question)

print(f"Question: {question}")
print(f"Answer: {answer}")

Question: What must interns do before leaving?
Answer: Interns must complete the following steps before leaving: inform their manager before submitting a formal resignation request, serve a 15-day notice period, and ensure that any emergency leave requests are approved by the reporting manager.


### Test Question 4: "What is the work from home policy?"

**Expected Answer:** Not found in context

**Why this test:**
- Tests how RAG handles out-of-scope questions
- Should gracefully admit when information isn't available
- Prevents hallucinations

In [35]:
question = "What is the work from home policy?"
answer = ask_rag_question(question)

print(f"Question: {question}")
print(f"Answer: {answer}")

Question: What is the work from home policy?
Answer: Not found in context.


---

# SECTION 12: RAG FAILURE CASES

## Understanding When RAG Doesn't Work

RAG is powerful but not perfect. Let's discuss common failure modes:

### 1. Retrieval Failure

**What happens:**
- The question embedding doesn't match any relevant chunks
- Retrieved chunks don't contain the answer
- LLM returns "Not found in context"

**Why it happens:**
- Vocabulary mismatch (question uses different words than document)
- Low-quality embeddings
- Insufficient context in retrieved chunks

**Example:**
- Question: "Do interns have dental coverage?"
- Document: "Interns receive no health insurance benefits"
- Embeddings miss the connection because "dental" and "health insurance" aren't similar enough

**Solution:**
- Use better embedding models
- Improve document content
- Adjust chunk size and overlap
- Use query expansion (ask multiple versions of the question)

---

### 2. Bad Chunking

**What happens:**
- Chunks split important information across boundaries
- Retrieved chunks are too small or too large
- Context is incomplete

**Why it happens:**
- Poor chunk size (too small loses context, too large loses precision)
- No chunk overlap
- Generic chunking strategy for specialized documents

**Example:**
- Fixed-size chunking splits "Notice period: 15 days" across two chunks
- Retrieved chunk only gets "Notice period:" → incomplete answer

**Solution:**
- Use recursive chunking with overlap
- Customize chunk size based on document type
- Use domain-specific chunking for specialized documents (code, legal, medical)

---

### 3. Context Pollution

**What happens:**
- Retrieved chunks contain irrelevant or contradictory information
- LLM gets confused by multiple conflicting sources
- Answer quality degrades

**Why it happens:**
- Similarity threshold too low
- Document contains outdated or conflicting policies
- Retrieved chunks address similar but different topics

**Example:**
- Question: "What's the laptop return policy?"
- Retrieved chunks: 1) New policy (15 days) 2) Old policy (30 days) 3) Unrelated IT policy
- LLM can't tell which is correct

**Solution:**
- Use higher similarity thresholds
- Maintain clean, consistent documents
- Use reranking to filter retrieved chunks (advanced)
- Implement document versioning

---

### 4. Hallucinations Despite RAG

**What happens:**
- Even with context, LLM makes up information
- Answer contains facts not in the retrieved chunks
- System violates the RAG principle of grounding

**Why it happens:**
- Weak prompt instructions
- LLM tries to be helpful and fills in gaps
- Context actually contradicts LLM's training

**Example:**
- Context: "Interns must give 15 days notice."
- LLM response: "Interns must give 15 days notice, and they receive a severance package."
- Second part is hallucinated (not in context)

**Solution:**
- Use strict prompts: "Answer ONLY using the context"
- Set temperature to 0 (deterministic)
- Use better models (more instruction-following)
- Implement output validation

---

## Key Takeaway

RAG is not magic. It's a system of components, each with potential failure modes.

**The RAG pipeline is only as good as:**
1. Document quality
2. Chunking strategy
3. Embedding quality
4. Similarity threshold
5. Prompt quality
6. LLM capability

In production, you monitor each component and optimize as needed.

---

# SECTION 13: 

## Advanced RAG Topics (Not Covered)

This course focused on the fundamentals. Here are advanced topics we intentionally skipped:

### Vector Stores (FAISS, Chroma, Pinecone)

**What they are:**
- Specialized databases for storing and searching embeddings
- Optimized for scale (millions of embeddings)

**Why we skipped them:**
- For learning RAG concepts, in-memory embeddings are sufficient
- They're optimization layers, not core concepts
- Add complexity for beginners

**When you need them:**
- Millions+ of documents
- Multi-user systems
- Production deployment
- Real-time updates

**Conceptually:**
```
Without Vector Store:
    Question Embedding → [Compare all chunk embeddings in memory] → Find similar

With Vector Store:
    Question Embedding → [Vector Store searches efficiently] → Find similar (fast!)
```

The retrieval logic doesn't change; only the search speed improves.

**Popular vector stores:**
- **FAISS**: Open-source, by Meta, great for research
- **Chroma**: Open-source, developer-friendly
- **Pinecone**: Cloud-based, fully managed

---

### Semantic Chunking (Advanced Topic)

**What it is:**
- Automatically detect semantic boundaries in text
- Split text where meaning naturally changes
- More sophisticated than fixed-size or sentence chunking

**Why we skipped it:**
- Requires additional models or heuristics
- Adds complexity
- Recursive chunking works well for most cases

**When you need it:**
- Specialized documents (code, legal, medical)
- Multiple topics in one document
- When recursive chunking doesn't preserve structure

---

## What About Reranking and Agents?

### Reranking
- Second-stage ranking of retrieved chunks
- Uses more expensive models to improve precision
- Useful when initial retrieval is noisy

### Agents
- LLMs that can use tools (APIs, search, etc.)
- Can reason about which information to retrieve
- Enables multi-step reasoning

**Why skipped:**
- Advanced concepts for intermediate learners
- Not needed for basic RAG
- Build fundamentals first

---

## What You Now Know

You have mastered:
- ✅ Fixed Size Chunking
- ✅ Sentence Chunking
- ✅ Recursive Chunking (recommended)
- ✅ Embeddings from OpenAI
- ✅ Cosine Similarity search
- ✅ The complete RAG pipeline
- ✅ Production-ready code

## Next Steps

To dive deeper:
1. Try implementing with different documents
2. Experiment with different chunk sizes
3. Test with vector stores (FAISS, Chroma)
4. Explore agents and multi-step reasoning
5. Study semantic chunking for specialized documents
6. Implement reranking for better accuracy

---

# Congratulations – You Built Your First End-to-End RAG System

## What You've Accomplished

In this 4-hour course, you built a complete RAG pipeline from scratch:

### ✅ Core Components Built

1. **PDF Loader**
   - Load and extract text from PDF documents
   - Handle multiple pages

2. **Text Extraction**
   - Convert binary PDF to usable text
   - Foundation for all downstream processing

3. **Fixed Size Chunking**
   - Simple, predictable chunking
   - Understand pros/cons of naive approaches

4. **Sentence Chunking**
   - Preserve semantic units
   - Better than fixed-size for most cases

5. **Recursive Chunking**
   - Industry-standard approach
   - Respects document structure
   - Controllable overlap for context preservation

6. **Embeddings**
   - Convert text to semantic vectors
   - Used OpenAI's text-embedding-3-small
   - 1536-dimensional vectors

7. **Cosine Similarity**
   - Mathematical measure of semantic similarity
   - Rank relevance of chunks
   - Used scikit-learn for computation

8. **Retrieval**
   - Find most relevant chunks for any question
   - Top-K retrieval (default: top 3)
   - Similarity-ranked results

9. **Context Construction**
   - Combine retrieved chunks into cohesive context
   - Format for LLM consumption

10. **Generation**
    - Use GPT-4o-mini to generate grounded answers
    - Answer based on retrieved context only
    - Prevent hallucinations with strict prompts

11. **Complete RAG Chatbot**
    - Modular design with reusable functions
    - `ask_pdf(question)` → grounded answer
    - Production-ready code

### ✅ Key Concepts Mastered

- Why RAG exists (LLMs don't know private documents)
- How chunking enables efficient retrieval
- Embedding as semantic representation
- Similarity search as core RAG mechanism
- Prompt engineering for reliability
- Testing and failure analysis

### ✅ Hands-On Skills

- Used OpenAI APIs (embeddings, chat completion)
- Worked with PDF extraction (pypdf)
- Text processing and chunking strategies
- Vector mathematics (cosine similarity)
- Prompt engineering for LLMs
- Built modular, reusable code

---

## The RAG Pipeline (What You Built)

```
┌─────────────────────────────────────────────────────┐
│                  RAG SYSTEM YOU BUILT                 │
├─────────────────────────────────────────────────────┤
│                                                       │
│  PDF Document                                         │
│       ↓                                               │
│  [PDF Reader] → Extract Text                          │
│       ↓                                               │
│  [Chunker] → Recursive Chunks (300 chars, 50 overlap) │
│       ↓                                               │
│  [Embedder] → 1536-dim vectors (text-embedding-3-small)│
│       ↓                                               │
│  [Knowledge Base] → Stored in memory                  │
│       ↓                                               │
│       ├─────────────────────────────────────┐         │
│       │                                     │         │
│       ▼                                     ▼         │
│   User Question                    Chunk Embeddings   │
│       │                                     │         │
│       └──→ [Embedder]                       │         │
│            │                                │         │
│            ▼                                │         │
│       Question Embedding                   │         │
│            │                                │         │
│            └──→ [Similarity Search]        │         │
│                 (cosine_similarity)        │         │
│                 │                           │         │
│                 ▼                           ▼         │
│            Similarity Scores → Top 3 Chunks         │
│                                │                     │
│                                ▼                     │
│                           [Context Builder]         │
│                                │                     │
│                                ▼                     │
│                       Question + Context             │
│                                │                     │
│                                ▼                     │
│                    [Prompt Engineer]                 │
│                                │                     │
│                                ▼                     │
│                        Formatted Prompt              │
│                                │                     │
│                                ▼                     │
│                     [GPT-4o-mini LLM]               │
│                                │                     │
│                                ▼                     │
│                       Grounded Answer                │
│                                │                     │
│                                ▼                     │
│                          User Gets Answer            │
│                                                       │
└─────────────────────────────────────────────────────┘
```

---

## From Here

### Short Term (Next Steps)
1. Try your RAG system with different PDFs
2. Experiment with chunk sizes (200, 400, 600 chars)
3. Test different queries to understand retrieval quality
4. Adjust top_k parameter (try 5, 10 chunks instead of 3)

### Medium Term (Build on This)
1. Add multiple documents to your knowledge base
2. Implement vector store (FAISS) for efficiency
3. Build a web interface (Flask, FastAPI)
4. Add source attribution (show which chunks were used)
5. Implement feedback loop to improve retrieval

### Long Term (Advanced Topics)
1. Semantic chunking for specialized documents
2. Reranking for improved accuracy
3. Agents with multi-step reasoning
4. Hybrid search (keyword + semantic)
5. Fine-tuned embedding models for your domain

---

## Key Takeaways

1. **RAG = Retrieval + Generation**
   - Retrieval finds relevant context
   - Generation creates grounded answers

2. **Embeddings are key**
   - Convert meaning to numbers
   - Enable semantic search

3. **Chunking matters**
   - Recursive chunking is recommended
   - Size and overlap are important

4. **Context is everything**
   - Quality of retrieved chunks determines quality of answer
   - Prompt quality affects reliability

5. **RAG is a system**
   - Every component matters
   - Monitor and optimize each part

---

## Thank You!

You've successfully completed the RAG course. You now understand:
- The complete RAG pipeline
- How to implement each component
